# Eksploracja: moduł konwersacji
Pełny pipeline: PDF → chunki → indeks → search → prompt → LLM → odpowiedź.
Testujemy wieloturową rozmowę z dokumentem.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "backend"))

from app.document.naive_processor import extract_pages, chunk_pages
from app.retrieval.vector_store import index_chunks, search
from app.conversation.prompt_builder import build_prompt
from app.conversation.llm_client import ask
from app.conversation.history import add_message, get_history

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Przygotowanie danych

In [2]:
pdf_path = str(project_root / "data" / "raport_2024_wybrane_strony.pdf")
pages = extract_pages(pdf_path)
chunks = chunk_pages(pages)
index_chunks(chunks)

print(f"Załadowano {len(pages)} stron, {len(chunks)} chunków zindeksowanych")

Załadowano 19 stron, 114 chunków zindeksowanych


## Pierwsze pytanie

In [3]:
question = "Czym zajmuje się Bank Gospodarstwa Krajowego?"

results = search(question)
system, messages = build_prompt(question, results)
answer = ask(messages, system)

add_message("demo", "user", question)
add_message("demo", "assistant", answer)

print(f"Q: {question}\n")
print(f"A: {answer}")

Q: Czym zajmuje się Bank Gospodarstwa Krajowego?

A: Na podstawie dostarczonego kontekstu, Bank Gospodarstwa Krajowego (BGK) zajmuje się następującymi działaniościami:

**Główna charakterystyka:**
"Bank Gospodarstwa Krajowego to państwowy bank rozwoju - jedna z pięciu największych takich instytucji w Europie. Od 100 lat wspieramy zrównoważony rozwój społeczno-gospodarczy Polski." [Strona 6]

**Kluczowe obszary działalności:**

1. **Wspieranie przedsiębiorczości** - bank "udoskonala ofertę wspierania przedsiębiorczości, także dzięki partnerstwom na poziomie międzynarodowym" [Strona 5]

2. **Działalność kredytowo-gwarancyjna** - "W 2024 roku zaangażowanie kredytowo-gwarancyjne BGK wyniosło 190 mld zł i było najwyższe w historii" [Strona 5]

3. **Wspieranie eksportu** - "Niezmiennie wspieramy polskie firmy przez finansowanie lub zabezpieczenia ekspansji. W 2024 roku do rynków zagranicznych, na których towarzyszymy polskim eksporterom, dołączyły Kostaryka i Salwador." [Strona 5]

4. **Oper

## Pytanie follow-up (z historią)

In [4]:
question2 = "Ilu pracowników zatrudnia?"

results2 = search(question2)
system2, messages2 = build_prompt(question2, results2, history=get_history("demo"))
answer2 = ask(messages2, system2)

add_message("demo", "user", question2)
add_message("demo", "assistant", answer2)

print(f"Q: {question2}\n")
print(f"A: {answer2}")

Q: Ilu pracowników zatrudnia?

A: Na podstawie dostarczonego kontekstu, Bank Gospodarstwa Krajowego zatrudnia:

**Na koniec 2024 roku:** 2 465 osób na 2 443,4 etatach [Strona 16]

**Struktura zatrudnienia według płci:**
- 1 494 kobiet (60,6%)
- 971 mężczyzn (39,4%) [Strona 16]

**Zmiana w stosunku do roku poprzedniego:**
- Na koniec 2023 roku: 2 320 pracowników
- Wzrost o 6,4% rok do roku [Strona 16]

**Podział według wymiaru czasu pracy (stan na 31.12.2024):**
- Pełen etat: 2 430 osób
- Część etatu: 35 osób [Strona 16]

**Podział według rodzaju umowy (stan na 31.12.2024):**
- Czas nieokreślony: 2 165 osób
- Czas określony: 300 osób [Strona 16]

Liczba 2 465 pracowników uwzględnia również "pracowników na urlopach rodzicielskich, wychowawczych oraz dłuższych zwolnieniach lekarskich" [Strona 16].


## Trzecie pytanie (test granic wiedzy)

In [5]:
question3 = "Jaki jest kurs dolara na dziś?"

results3 = search(question3)
system3, messages3 = build_prompt(question3, results3, history=get_history("demo"))
answer3 = ask(messages3, system3)

add_message("demo", "user", question3)
add_message("demo", "assistant", answer3)

print(f"Q: {question3}\n")
print(f"A: {answer3}")

Q: Jaki jest kurs dolara na dziś?

A: Na podstawie dostarczonego kontekstu nie mogę odpowiedzieć na pytanie o aktualny kurs dolara. Dostarczony dokument zawiera informacje o działalności Banku Gospodarstwa Krajowego, ale nie zawiera żadnych danych o bieżących kursach walutowych.

Aby uzyskać informacje o aktualnym kursie dolara, należałoby skorzystać z innych źródeł, takich jak strony banków, serwisy finansowe lub NBP.


## Podgląd: co widzi LLM?

In [6]:
print("=== SYSTEM ===")
print(system3)
print()
for i, msg in enumerate(messages3):
    print(f"=== MESSAGE {i} ({msg['role']}) ===")
    print(msg["content"][:300])
    if len(msg["content"]) > 300:
        print(f"... ({len(msg['content'])} znaków)")
    print()

=== SYSTEM ===
You are a helpful assistant answering questions based on provided document excerpts. Rules:
- Answer ONLY based on the provided context. Do not use outside knowledge.
- Quote the source text verbatim when possible.
- Always mention the page number: [Strona X].
- If the context does not contain enough information to answer, say so explicitly.
- Answer in the same language as the question.

=== MESSAGE 0 (user) ===
Document context:

[Strona 8]: "go Funduszu Drogowego 
- w sumie na kwotę
3,5 mld USD. Łączny
popyt na obie serie obligacji
dolarowych ukształtował się
na rekordowym poziomie
6,6 mld USD.
BGK członkiem PCAF
Dołączyliśmy do Partnership for 
Carbon Accounting Financials – jako 
piąty bank z Polski.  

... (1875 znaków)

=== MESSAGE 1 (user) ===
Czym zajmuje się Bank Gospodarstwa Krajowego?

=== MESSAGE 2 (assistant) ===
Na podstawie dostarczonego kontekstu, Bank Gospodarstwa Krajowego (BGK) zajmuje się następującymi działaniościami:

**Główna charakterystyka:**
"B